# Projet Hadoop — Entraînement et résultats du modèle

Ce notebook présente la préparation des données, le modèle K-Means entraîné, ses métriques et quelques cas de test.

**Modèle final :** K-Means, `k=8`, `seed=42`, 26 features.

## 1. Initialisation de Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, round as spark_round
from pyspark.ml.clustering import KMeansModel
from pyspark.ml.feature import StandardScalerModel
import pandas as pd
import json
import matplotlib.pyplot as plt

HDFS_PATH = "hdfs://localhost:9000/user/user/data"

spark = (
    SparkSession.builder
    .appName("Notebook - Projet séries TV")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark initialisé")

## 2. Chargement des résultats du clustering

In [ ]:
df_clustered = spark.read.parquet(f"{HDFS_PATH}/shows_clustered")
print("Nombre de séries :", df_clustered.count())
df_clustered.show(10, truncate=False)

## 3. Rapport du modèle final

In [ ]:
with open("outputs/model_report.json", encoding="utf-8") as f:
    report = json.load(f)

report

Le score de silhouette mesure la séparation des clusters. Plus il est proche de 1, mieux les groupes sont séparés.

## 4. Taille des clusters

In [ ]:
cluster_sizes = (
    df_clustered.groupBy("cluster_id")
    .count()
    .orderBy("cluster_id")
)
cluster_sizes.show()

In [ ]:
cluster_sizes_pd = cluster_sizes.toPandas()
plt.figure(figsize=(8, 4))
plt.bar(cluster_sizes_pd["cluster_id"].astype(str), cluster_sizes_pd["count"])
plt.xlabel("Cluster")
plt.ylabel("Nombre de séries")
plt.title("Répartition des séries par cluster")
plt.show()

## 5. Résumé des clusters

In [ ]:
summary = (
    df_clustered.groupBy("cluster_id")
    .agg(
        count("*").alias("nb_series"),
        spark_round(avg("popularity"), 2).alias("popularite_moyenne"),
        spark_round(avg("vote_quality"), 2).alias("qualite_vote_moyenne"),
        spark_round(avg("vote_count"), 2).alias("votes_moyens"),
        spark_round(avg("serie_length"), 2).alias("longueur_moyenne"),
        spark_round(avg("content_density"), 2).alias("episodes_par_saison")
    )
    .orderBy("cluster_id")
)
summary.show(truncate=False)

## 6. Comparaison des expériences

In [ ]:
results = pd.read_csv("experiments/results.csv")
results

In [ ]:
mean_results = results.groupby("k", as_index=False).agg(
    silhouette=("silhouette", "mean"),
    cout=("cout", "mean")
)

plt.figure(figsize=(8, 4))
plt.plot(mean_results["k"], mean_results["silhouette"], marker="o")
plt.xlabel("Nombre de clusters K")
plt.ylabel("Silhouette moyenne")
plt.title("Silhouette moyenne selon K")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(mean_results["k"], mean_results["cout"], marker="o")
plt.xlabel("Nombre de clusters K")
plt.ylabel("Coût moyen")
plt.title("Coût K-Means selon K")
plt.show()

## 7. Vérification du chargement des modèles sauvegardés

In [ ]:
scaler_model = StandardScalerModel.load(f"{HDFS_PATH}/models/scaler_model")
kmeans_model = KMeansModel.load(f"{HDFS_PATH}/models/kmeans_model")

print("Scaler chargé")
print("K-Means chargé")
print("Nombre de clusters :", kmeans_model.getK())

## 8. Cas de test

In [ ]:
series_test = ["Lie to Me", "Sherlock", "Breaking Bad", "Friends"]

df_clustered.filter(col("name").isin(series_test)) \
    .select("name", "cluster_id", "popularity", "vote_quality", "vote_count") \
    .orderBy("name") \
    .show(truncate=False)

## 9. Conclusion

- Le modèle final a été entraîné sur 53 532 séries et 26 features.
- Le meilleur compromis retenu est `K=8`, `seed=42`.
- Le modèle et le scaler sont sauvegardés dans HDFS.
- Les modèles peuvent être rechargés et utilisés sans nouvel entraînement.
- Les clusters sont déséquilibrés, ce qui reflète les limites du dataset et des features disponibles.

In [ ]:
# À exécuter à la fin du notebook
spark.stop()